# Interactive use of cleanest functionality

<div style="
    padding: 10px;
    border-left: 5px solid #f39c12;
    background-color: #fcf3cf;
">
<strong>Warning:</strong> 
In this notebook we are not trying to demonstrate the usefulness of a particular detection algorithm nor the benefits of using a particular interpolation method to replace cosmic-ray pixels. We are simply showing the basic Python code required to use programmatically some of the functionality available in <strong>tea-cleanest</strong>.
</div>
<br>

In [ ]:
from astropy.io import fits
from datetime import datetime
import numpy as np
from scipy import ndimage

import teareduce as tea
from teareduce import cleanest

In [ ]:
time_ini = datetime.now()

## Import specific packages

The following code checks whether some specific packages devoted to detecting and cleaning cosmic rays have already been installed. This is not necessary if you have previously successfully installed and run **tea-cleanest**.

In [ ]:
import importlib

# Map: pip package name -> Python import name
required = {
    "ccdproc": "ccdproc",
    "maskfill": "maskfill",     # change if the import name differs
    "deepCR": "deepCR",         # case sensitive; adjust if module is 'deepcr'
    "cosmic-conn": "cosmic_conn"  # hyphen in pip, underscore in import (example)
}

missing = []

for pip_name, import_name in required.items():
    print(f"Checking availability of package: {import_name}")
    try:
        importlib.import_module(import_name)
    except ModuleNotFoundError:
        missing.append((pip_name, import_name))

if missing:
    msg_lines = ["This notebook requires the following packages:"]
    for pip_name, import_name in missing:
        msg_lines.append(
            f" - import name: '{import_name}' (install with: pip install {pip_name})"
        )
    raise ModuleNotFoundError("\n".join(msg_lines))

# Import all packages into the global namespace
for import_name in required.values():
    print(f"Importing: {import_name}")
    globals()[import_name] = importlib.import_module(import_name)

In [ ]:
try:
    import PyCosmic
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "The 'teareduce.cleanest' module requires the 'PyCosmic' package.\n"
        "Please install this module using:\n"
        "`pip install git+https://github.com/nicocardiel/PyCosmic.git@test`"        
    ) from e

print("Importing: PyCosmic")

## Example data

Download the example data files used in this notebook.

In [ ]:
for fname in [
    'notebooks/cleanest/examplecr1.fits',
    'notebooks/cleanest/examplecr2.fits',
    'notebooks/cleanest/examplecr3.fits',
    'notebooks/cleanest/examplecr1_cleaned_with_surface.fits',
]:
    tea.get_cookbook_file(fname)

## Detect CRs in single image using L.A.Cosmic

The following lines of code show how to run the cleaning of the first example image used in [Simple execution (single image)](https://nicocardiel.github.io/teareduce-cookbook/docs/cleanest/cleanest.html#simple-execution-single-image). The results will be compared with a previously computed cleaned image, which serves as a test of the code.

In [ ]:
# Read the input file
with fits.open('examplecr1.fits') as hdul:
    data = hdul[0].data

### Running L.A.Cosmic only once

As explained in the documentation of **tea-cleanest**, the L.A.Cosmic algorithm can be run twice (`run 1` and `run 2`) on the same image using different values for some parameters. Before dealing with this case, let's illustrate how to run L.A.Cosmic simply once, using the same parameters used in `run 1`.

Here we are using the `cleanest.lacosmicpad()` function, which is a wrapper around the `cosmicray_lacosmic()` function from the **ccdproc** package. This function pads the input image array before applying L.A.Cosmic (this helps the algorithm locate CR pixels close to the image borders). If `inbkg` or `invar` arrays are provided, they are also padded accordingly. After processing, the padding is removed to return an array of the original size.

Users should consult the [cosmicray_lacosmic()](https://ccdproc.readthedocs.io/en/latest/api/ccdproc.cosmicray_lacosmic.html) function documentation provided by the ccdproc package.

In [ ]:
pad_width = 10

In [ ]:
cleandata_lacosmic, mask_crfound = cleanest.lacosmicpad(
    pad_width=pad_width,
    ccd=data,
    gain=1.0,
    readnoise=6.5,
    sigclip=5.0,
    sigfrac=0.3,
    objlim=5.0,
    niter=4,
    fsmode='median',
    verbose=True,
)

29 cosmic pixels this iteration
Iteration 4:


17 cosmic pixels this iteration


Check image shapes

In [ ]:
data.shape, cleandata_lacosmic.shape, mask_crfound.shape

### Clean the detected CRs

Using the cosmic-ray pixel mask, we can proceed to interpolate the cosmic rays using the desired method. 

The cosmic-ray pixels are initially dilated by the specified number of pixels (0 means no dilation). The resulting flagged pixels are grouped into cosmic-ray features. Each cosmic-ray feature is then interpolated using the specified interpolation method.


Note that the interpolation methods `lacosmic` and `auxfile`, available in the interactive use of **tea-cleanest** are not implemented in this function because both cases are simply an inmediate replacement of the cosmic ray pixels in the data array by the corresponding pixels in another array using the mask array. Therefore, these two methods do not require any interpolation algorithm.

In the following example we are replacing the detected pixels using the surface interpolation (`interp_method='s'`).

In [ ]:
data_cleaned, masked = cleanest.interpolate(data, mask_crfound, interp_method='s', npoints=2, degree=1, debug=True)

For a more detailed explanation of the available possibilities, have a look at the documentation of this function.

In [ ]:
?cleanest.interpolate

### Running L.A.Cosmic twice

Although the prior execution of L.A.Cosmic has allowed the removal of many cosmic rays in the initial image, visual inspection reveals that the tails of some cosmic rays (i.e., pixels with weaker but still affected signals) have not been interpolated. The way **tea-cleanest** addresses this situation is by running L.A.Cosmic a second time, using a lower threshold to detect cosmic rays. To prevent the appearance of many false positives, the code groups suspected pixels into cosmic-ray features. The newly detected cosmic-ray features from the second run are only retained if they are in contact with cosmic-ray features already identified in the first execution of L.A.Cosmic.

In [ ]:
cleandata_lacosmic2, mask_crfound2 = cleanest.lacosmicpad(
    pad_width=pad_width,
    ccd=data,
    gain=1.0,
    readnoise=6.5,
    sigclip=3.0,      # modified
    sigfrac=0.3,
    objlim=5.0,
    niter=4,
    fsmode='median',
    verbose=True,
)

In [ ]:
np.sum(mask_crfound), np.sum(mask_crfound2)

The merging of both masks, following the strategy previously mentioned, is carried out by the auxiliary function `merge_peak_tail_masks()`.

In [ ]:
mask_crfound = cleanest.merge_peak_tail_masks(
    mask_peaks=mask_crfound, 
    mask_tails=mask_crfound2,
    verbose=True
)

We then proceed to interpolate the cosmic-ray pixels using the new mask.


In [ ]:
data_cleaned, masked = cleanest.interpolate(
    data=data, 
    mask_crfound=mask_crfound, 
    interp_method='s', 
    npoints=2, 
    degree=1, 
    debug=True
)

### Comparison with reference image

One of the downloaded images at the beginning of the execution of this notebook is the file `examplecr1_cleaned_with_surface.fits`, which was generated interactively using **tea-cleanest**. We can compare this file with the result from the last section.

In [ ]:
reference_filename = "examplecr1_cleaned_with_surface.fits"
with fits.open(reference_filename) as hdul:
    reference_data = hdul[0].data
    reference_mask = hdul["CRMASK"].data

Check data array

In [ ]:
np.array_equal(data_cleaned, reference_data)

Check CR mask array

In [ ]:
np.array_equal(masked, reference_mask)

## Using other CR detection algorithm

You can choose any suitable detection algorithm before making use of the `cleanest.interpolation()` function. Let's see how to do it using other algorithms incorporated in **tea-cleanest**.

### PyCosmic

In this case one can proceed by calling directly the `det_cosmics()` function available in the [PyCosmic package](https://github.com/brandherd/PyCosmic/tree/master).

In [ ]:
?PyCosmic.det_cosmics

In [ ]:
out = PyCosmic.det_cosmics(
    data=data,
    sigma_det=5.0,
    rlim=1.2,
    iterations=5,
    fwhm_gauss=[2.5, 2.5],
    replace_box=[5, 5],
    replace_error=1e6,
    increase_radius=0,
    gain=1.0,
    rdnoise=6.5,
    bias=0,
    verbose=True,
)

Although the previous function already returns a cleaned version of the image in `out.data`, the use of `cleanest.interpolation()` allows you to use simply the mask stored in `out.mask` and select any of the available interpolation algorithms in the latter function.

In [ ]:
mask_crfound = out.mask.astype(bool)

In [ ]:
data_cleaned, masked = cleanest.interpolate(
    data=data, 
    mask_crfound=mask_crfound,
    interp_method='s', 
    npoints=2, 
    degree=1, 
    debug=True
)

### deepCR

Here we make use of two functions in the [deepCR package](https://deepcr.readthedocs.io/en/latest/)

In [ ]:
?deepCR.deepCR

In [ ]:
# Initialize the deepCR model
mdl = deepCR.deepCR(mask="ACS-WFC")

In [ ]:
?mdl.clean

In [ ]:
mask_crfound, cleandata_deepcr = mdl.clean(
    img0=data,
    threshold=0.5,
    inpaint=True,
)

Although the previous function already returns a cleaned version of the image in `cleandata_deepcr`, the use of `cleanest.interpolation()` allows you to use simply the mask stored in `mask_crfound` and select any of the available interpolation algorithms in the latter function.

In [ ]:
data_cleaned, masked = cleanest.interpolate(
    data=data, 
    mask_crfound=mask_crfound,
    interp_method='s', 
    npoints=2, 
    degree=1, 
    debug=True
)

### Cosmic-CoNN

Here we use the functionality available in the [cosmic_conn package](https://cosmic-conn.readthedocs.io/en/latest/index.html).

In [ ]:
# Initialize the generic ground-imaging model
cr_model = cosmic_conn.init_model("ground_imaging")

If we get a warning message here is because the installed cosmic_conn package is using an old PyTorch function (torch.set_default_tensor_type()) that has been deprecated in PyTorch 2.1+. The PyTorch developers want people to use the newer functions torch.set_default_dtype() and torch.set_default_device() instead.

In [ ]:
# The model outputs a CR probability map
print(f"Running Cosmic-CoNN version: {cosmic_conn.__version__}  (please wait...)")
cr_prob = cr_model.detect_cr(data.astype(np.float32))
print("Done!")

In [ ]:
# Set probability threshold
threshold = 0.5

In [ ]:
mask_crfound = cr_prob > threshold

In this case we only have a detection mask and not a cleaned version of the data. To obtain a cleaned version, we need to use `cleanest.interpolate()`.

In [ ]:
data_cleaned, masked = cleanest.interpolate(
    data=data, 
    mask_crfound=mask_crfound,
    interp_method='s', 
    npoints=2, 
    degree=1, 
    debug=True
)

## Detect CRs in multiple equivalent exposures

When several equivalent exposures are available, one can try to clean each of them making use of the information from the remaining images. 

### Two equivalent exposures

For example, when considering only two exposures, one can detect CRs in one image and replace the suspected pixels using information from the second image. The roles of both images can be interchanged to clean the second image using information from the first exposure. The basic assumption behind this strategy is that the same pixels have not been hit by cosmic rays in both exposures, which may not be the case for long exposure times.

Let's illustrate this approach using `examplecr1.fits` and `examplecr2.fits`.

In [ ]:
with fits.open('examplecr1.fits') as hdul:
    data1 = hdul[0].data

with fits.open('examplecr2.fits') as hdul:
    data2 = hdul[0].data

We are going to illustrate the procedure using the L.A.Cosmic procedure, which is relatively fast.

In [ ]:
pad_width = 10

TBD

In [ ]:
tea.elapsed_time_since(time_ini)